In [2]:
!nvidia-smi

Thu Jan 29 15:12:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    %pip install unsloth vllm
else:
    %pip install --upgrade -qqq uv
    !uv pip install vllm==0.11.2 unsloth-zoo unsloth
    !uv pip install transformers==4.56.2
    !uv pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastVisionModel
import torch

model_id = 'unsloth/Qwen2.5-VL-3B-Instruct'
max_seq_length = 16384 # Must be this long for VLMs
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    # fast_inference = True, # Enable vLLM fast inference
    fast_inference = False, # Disable to fix the vLLM bug on T4
    gpu_memory_utilization = 0.8, # Reduce if out of memory
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
ERROR 01-29 15:13:53 [fa_utils.py:64] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Qwen2_5_Vl patching. Transformers: 4.56.2. vLLM: 0.11.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.79G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [ ]:
from datasets import load_dataset

data_id = 'AI4Math/MathVista'
data_split = 'testmini'
dataset = load_dataset(data_id, split=data_split)

README.md: 0.00B [00:00, ?B/s]

data/testmini-00000-of-00001-725687bf7a1(…):   0%|          | 0.00/142M [00:00<?, ?B/s]

data/test-00000-of-00002-6b81bd7f7e2065e(…):   0%|          | 0.00/358M [00:00<?, ?B/s]

data/test-00001-of-00002-6a611c71596db30(…):   0%|          | 0.00/386M [00:00<?, ?B/s]

Generating testmini split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5141 [00:00<?, ? examples/s]

In [6]:
def is_numeric_answer(example):
    try:
        float(example['answer'])
        return True
    except:
        return False

def resize_images(example):
    image = example['decoded_image']
    image = image.resize((512, 512))
    example['decoded_image'] = image
    return example

def convert_to_rgb(example):
    image = example['decoded_image']
    if image.mode != 'RGB':
        image = image.convert('RGB')
    example['decoded_image'] = image
    return example

dataset = dataset.filter(is_numeric_answer) # Keep only numeric answers
dataset = dataset.map(resize_images) # Resize to (512, 512)
dataset = dataset.map(convert_to_rgb) # Then convert to RGB

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/566 [00:00<?, ? examples/s]

Map:   0%|          | 0/566 [00:00<?, ? examples/s]

In [7]:
# Define the delimiter variables for clarity and easy modification
REASONING_START = "<REASONING>"
REASONING_END = "</REASONING>"
SOLUTION_START = "<SOLUTION>"
SOLUTION_END = "</SOLUTION>"

def make_conversation(example):
    # Define placeholder constants if they are not defined globally
    # The user's text prompt
    text_content = (
        f"{example['question']}. Also first provide your reasoning or working out"\
        f" on how you would go about solving the question between {REASONING_START} and {REASONING_END}"
        f" and then your final answer between {SOLUTION_START} and (put a single float here) {SOLUTION_END}"
    )

    # Construct the prompt in the desired multi-modal format
    prompt = [
        {
            'role': 'user',
            'content': [
                {'type': 'image'},  # Placeholder for the image
                {'type': 'text', 'text': text_content},  # The text part of the prompt
            ],
        },
    ]
    # The actual image data is kept separate for the processor
    return {'prompt': prompt, 'image': example['decoded_image'], 'answer': example['answer']}

train_dataset = dataset.map(make_conversation)

# We're reformatting dataset like this because decoded_images are the actual images
# The 'image': example['decoded_image'] does not properly format the dataset correctly

# 1. Remove the original 'image' column
train_dataset = train_dataset.remove_columns('image')

# 2. Rename 'decoded_image' to 'image'
train_dataset = train_dataset.rename_column('decoded_image', 'image')

train_dataset = train_dataset.map(
    lambda example: {
        'prompt': tokenizer.apply_chat_template(
            example['prompt'],
            tokenize = False,
            add_generation_prompt = True, # Must add assistant
        )
    }
)

Map:   0%|          | 0/566 [00:00<?, ? examples/s]

Map:   0%|          | 0/566 [00:00<?, ? examples/s]

In [8]:
# from vllm import SamplingParams

# sampling_params = SamplingParams(
#     temperature = 1.0,
#     top_k = 50,
#     max_tokens = 1024,
# )

# outputs = model.fast_generate(
#     {
#         "prompt": train_dataset[100]["prompt"],
#         "multi_modal_data": {"image": train_dataset[100]["image"]}
#     },
#     sampling_params,
# )
# print(outputs[0].outputs[0].text)

In [9]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(model_name)

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [25]:
step = 100
start_idx = 0
end_idx = min(start_idx+step, len(train_dataset))

inputs = processor.image_processor(images=list(train_dataset['image'][start_idx:end_idx]), return_tensors='pt')
pixel_values = inputs['pixel_values'].to(model.device)
image_grid_thw = inputs['image_grid_thw'].to(model.device)

with torch.no_grad():
    vision_embs = model.visual(pixel_values, image_grid_thw)

print(f"Range: {start_idx}-{end_idx-1}")
print("Pixel values shape:", pixel_values.shape)
print("Image grid shape:", image_grid_thw.shape)
print("Vision embeddings shape:", vision_embs.shape)

Range: 0-99
Pixel values shape: torch.Size([129600, 1176])
Image grid shape: torch.Size([100, 3])
Vision embeddings shape: torch.Size([32400, 2048])


In [26]:
batch = vision_embs # (N, 2048)

# Statistics for Standardization
batch_n = batch.shape[0]
batch_sum = batch.sum(dim=0)
batch_sum_sq = (batch**2).sum(dim=0)

# Statistics for Min-Max Normalization
batch_min = batch.min(dim=0).values
batch_max = batch.max(dim=0).values

print("Batch size:", batch_n)
print("Batch sum shape:", batch_sum.shape)
print("Batch sum squared shape:", batch_sum_sq.shape)
print("Batch min shape:", batch_min.shape)
print("Batch max shape:", batch_max.shape)

Batch size: 32400
Batch sum shape: torch.Size([2048])
Batch sum squared shape: torch.Size([2048])
Batch min shape: torch.Size([2048])
Batch max shape: torch.Size([2048])


In [27]:
import os
from pathlib import Path
import numpy as np

save_dir = Path(f'model_{model_id.replace("/", "_")}.data_{data_id.replace("/", "_")}_{data_split}')
os.makedirs(save_dir, exist_ok=True)

range_tag = f'{start_idx}-{end_idx-1}'
vision_data_path = save_dir / f'{range_tag}.vision_data.npz'
stats_path = save_dir / f'{range_tag}.stats.npz'

np.savez(
    vision_data_path,
    pixel_values=pixel_values.cpu().numpy(),
    image_grid_thw=image_grid_thw.cpu().numpy(),
    vision_embs=vision_embs.cpu().numpy()
)
np.savez(
    stats_path,
    batch_n=batch_n,
    batch_sum=batch_sum.cpu().numpy(),
    batch_sum_sq=batch_sum_sq.cpu().numpy(),
    batch_min=batch_min.cpu().numpy(),
    batch_max=batch_max.cpu().numpy()
)

print("Vision data saved to:", vision_data_path)
print("Statistics saved to:", stats_path)

Vision data saved to: model_unsloth_Qwen2.5-VL-3B-Instruct.data_AI4Math_MathVista_testmini/0-99.vision_data.npz
Statistics saved to: model_unsloth_Qwen2.5-VL-3B-Instruct.data_AI4Math_MathVista_testmini/0-99.stats.npz


In [32]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path=str(save_dir),
    path_in_repo=str(save_dir),
    repo_id='alxxtexxr/VRBI-Data',
    repo_type='dataset'
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...a_testmini/0-99.stats.npz: 100%|##########| 17.7kB / 17.7kB            

  ...mini/0-99.vision_data.npz:   3%|3         | 25.1MB /  742MB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/VRBI-Data/commit/d575f61e31de82aa9f84db87cff85acb39467676', commit_message='Upload folder using huggingface_hub', commit_description='', oid='d575f61e31de82aa9f84db87cff85acb39467676', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/VRBI-Data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/VRBI-Data'), pr_revision=None, pr_num=None)